# Media Bias Detection — Exploratory Data Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

STOP_WORDS = set(stopwords.words('english')) - {'not', 'no', 'but', 'however'}
df = pd.read_csv('../data/sample_headlines.csv')
print(df.shape)
df.head()

## 1. Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2166ac', '#4dac26', '#d01c8b']
df['label'].value_counts().plot(kind='bar', color=colors, ax=ax, edgecolor='white')
ax.set_title('Class Distribution — Bias Labels', fontsize=14, fontweight='bold')
ax.set_xlabel('Bias Label')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Headline Length Distribution by Class

In [ ]:
df['word_count'] = df['headline'].str.split().str.len()
fig, ax = plt.subplots(figsize=(9, 5))
for label, color in zip(['left', 'neutral', 'right'], ['#2166ac', '#4dac26', '#d01c8b']):
    subset = df[df['label'] == label]['word_count']
    ax.hist(subset, bins=10, alpha=0.6, label=label.capitalize(), color=color, edgecolor='white')
ax.set_title('Headline Length Distribution by Bias Class', fontsize=13, fontweight='bold')
ax.set_xlabel('Word Count')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()
print(df.groupby('label')['word_count'].describe())

## 3. Top Words Per Bias Class

In [ ]:
import re, string

def get_top_words(df, label, n=12):
    texts = ' '.join(df[df['label'] == label]['headline'].str.lower())
    texts = texts.translate(str.maketrans('', '', string.punctuation))
    tokens = [t for t in texts.split() if t not in STOP_WORDS and len(t) > 2]
    return Counter(tokens).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (label, color) in zip(axes, [('left', '#2166ac'), ('neutral', '#4dac26'), ('right', '#d01c8b')]):
    top = get_top_words(df, label)
    words, counts = zip(*top)
    ax.barh(words[::-1], counts[::-1], color=color, edgecolor='white')
    ax.set_title(f'Top Words — {label.capitalize()}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Frequency')
plt.suptitle('Most Frequent Words by Bias Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Word Co-occurrence Heatmap

In [ ]:
import numpy as np
bias_words = ['republican', 'democrat', 'policy', 'crisis', 'bill', 'tax', 
              'workers', 'corporate', 'radical', 'senate', 'climate', 'economy']

def cooccurrence_matrix(df, words):
    matrix = np.zeros((len(words), len(words)))
    for _, row in df.iterrows():
        tokens = set(row['headline'].lower().split())
        for i, w1 in enumerate(words):
            for j, w2 in enumerate(words):
                if i != j and w1 in tokens and w2 in tokens:
                    matrix[i, j] += 1
    return matrix

comat = cooccurrence_matrix(df, bias_words)
plt.figure(figsize=(10, 8))
sns.heatmap(pd.DataFrame(comat, index=bias_words, columns=bias_words),
            cmap='YlOrRd', annot=True, fmt='.0f', linewidths=0.5)
plt.title('Word Co-occurrence Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Bigram Analysis

In [ ]:
from nltk.util import ngrams

def get_bigrams(df, label, n=10):
    texts = ' '.join(df[df['label'] == label]['headline'].str.lower())
    texts = texts.translate(str.maketrans('', '', string.punctuation))
    tokens = [t for t in texts.split() if t not in STOP_WORDS]
    bg = list(ngrams(tokens, 2))
    return Counter(bg).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (label, color) in zip(axes, [('left', '#2166ac'), ('neutral', '#4dac26'), ('right', '#d01c8b')]):
    top = get_bigrams(df, label, 8)
    phrases = [' '.join(bg) for bg, _ in top]
    counts = [c for _, c in top]
    ax.barh(phrases[::-1], counts[::-1], color=color)
    ax.set_title(f'Top Bigrams — {label.capitalize()}', fontsize=11, fontweight='bold')
plt.suptitle('Top Bigrams by Bias Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()